# Straighten the scans

There are six scans that are crooked and have to be straightened. By the end of this notebook you
will have straightened them with a small convolutional neural network that you trained yourself,
on data you made yourself from pages known to be straight. The six scans come from documents the
network never sees in training.

**Keep your work.** In Colab, choose *File → Save a copy in Drive* first: changes to the copy
opened from GitHub are not saved.

**How to work.** Run one cell at a time with Shift+Enter, make adjustments, experiment.

In [ ]:
#@title The pages, packed as text. Run this cell; there is no need to open it.
PACKED = "..."


In [ ]:
# The pages travel inside this notebook: the cell above holds a zip file, written out as text.
# This cell turns the text back into files, in a folder next to the notebook.
import base64, pathlib, zipfile
zip_bytes = base64.b64decode(PACKED)
pathlib.Path("workshop-pages.zip").write_bytes(zip_bytes)
zipfile.ZipFile("workshop-pages.zip").extractall("workshop-pages")
ROOT = pathlib.Path("workshop-pages")
PAGES = ROOT / "pages"
CROOKED = ROOT / "crooked"
straight = sorted(PAGES.glob("*.jpg"))
print("straight pages:", len(straight))
crooked = sorted(CROOKED.glob("scan-*.jpg"))
print("crooked scans:", len(crooked))


In [ ]:
import math
import random
import time
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch

%config InlineBackend.figure_format = "retina"   # sharp pictures on high-resolution screens

SEED = 2026
random.seed(SEED)

def show(images, titles=(), cols=6, width=3):
    """Grey images in a grid of up to cols in a row; 0 is black, 255 is white."""
    cols = min(cols, len(images))
    rows = math.ceil(len(images) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(width * cols, width * 1.35 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for i in range(len(images)):
        axes.flat[i].imshow(images[i], cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        if i < len(titles):
            axes.flat[i].set_title(titles[i], fontsize=9)
    plt.show()

def turn(image, angle):
    """The image turned by angle degrees, anticlockwise when positive; it grows so that nothing is cut off, the new corners white."""
    turned = Image.fromarray(image).rotate(angle, resample=Image.Resampling.BICUBIC, expand=True, fillcolor=255)
    return np.asarray(turned)

## 1. The job: six crooked scans

In [ ]:
scans = pd.read_csv(CROOKED / "scans.csv")
scan_images = [np.asarray(Image.open(CROOKED / file)) for file in scans["file"]]
show(scan_images, titles=list(scans["file"]), cols=3, width=4.4)

Try one by hand: type an angle, look, try again.

In [ ]:
scan = scan_images[0]   # set this index to 0, 1, 2, 3, 4 or 5 to load the different scans
angle = 0.0      # degrees; positive turns the scan anticlockwise. Change it until the lines are level.
show([scan, turn(scan, angle)], titles=["the scan as it is", f"turned by {angle:+.1f}°"], width=6.5)

## 2. Pages known to be straight

These pages were checked: their lines run level.

In [ ]:
pages = pd.read_csv(PAGES / "pages.csv")
train = pages[pages["split"] == "train"]
validation = pages[pages["split"] == "validation"]
print(len(train), "pages to learn from,", len(validation), "to check on, from", pages["document"].nunique(), "documents")
pages.head()   # the first five rows of the table (a pandas DataFrame)

In [ ]:
page_images = [np.asarray(Image.open(PAGES / file)) for file in pages["file"]]
titles = [document + "\n" + split for document, split in zip(pages["document"], pages["split"])]
show(page_images, titles=titles, width=1.9)

## 3. Make the training data yourself

Nobody has to measure angles and create training data manually. You take a page known to be
straight, turn it by an angle you choose, and cut a small square from inside the text: the square
is one example, the angle is its label. Thousands of such squares, from all the straight pages,
show the network what text looks like at every angle.

Every page is first shrunk to a quarter of its height and width: 1200 pixels high become 300. A
square of 80 pixels then holds between four and ten lines of writing, depending on the size of
the hand, so the network learns the angle of large and small writing alike.

In [ ]:
SCALE = 0.25     # every page shrunk to a quarter of its height and width
CROP = 80        # pixels: the side of one example
MAX_ANGLE = 15   # degrees, either way

def scaled_page(page, folder):
    """The page from the folder, shrunk by SCALE, and its text box, shrunk the same way."""
    image = Image.open(folder / page["file"])
    image = image.resize((round(image.width * SCALE), round(image.height * SCALE)))
    box = (round(page["left"] * SCALE), round(page["top"] * SCALE), round(page["right"] * SCALE), round(page["bottom"] * SCALE))
    return np.asarray(image), box

page = train.iloc[0]
image, box = scaled_page(page, PAGES)
print(page["document"], "shrunk to", image.shape[1], "×", image.shape[0], "pixels; text box", box)
show([image], titles=[page["document"]], width=3)

In [ ]:
# One example, step by step: cut a square a little larger than needed, turn it, keep the middle.
# Turning 100 pixels by up to 15° leaves the middle 80 × 80 free of empty corners.
MARGIN = 10              # pixels on each side that the turning may spoil
CUT = CROP + 2 * MARGIN  # 100: the side of the square cut before turning
left, top, right, bottom = box
x = random.randint(left, right - CUT)   # the square's top-left corner, anywhere inside the text box
y = random.randint(top, bottom - CUT)
angle = 9.0
square = image[y:y + CUT, x:x + CUT]
turned = turn(square, angle)
edge = (len(turned) - CROP) // 2   # turning made the square larger: the middle starts this far in
crop = turned[edge:edge + CROP, edge:edge + CROP]
show([square, turned, crop],
     titles=[f"{CUT} × {CUT} pixels, cut", f"turned by {angle:+.1f}°", f"the middle {CROP} × {CROP}: label {angle:+.1f}°"],
     width=3)

## 4. Build the set

The cell below does the same 400 times for each page, each time at a new spot and at a new angle
drawn at random between -15° and +15°. The pages to learn from give the examples the network
trains on; the six validation pages give examples it never trains on.

In [ ]:
PER_PAGE = 400   # examples from each page

def make_examples(pages):
    """PER_PAGE crops from every page of the table, at random spots and angles, and their angles."""
    n = len(pages) * PER_PAGE
    crops = np.zeros((n, CROP, CROP), dtype=np.uint8)
    angles = np.zeros(n, dtype=np.float32)
    i = 0
    for p in range(len(pages)):
        image, box = scaled_page(pages.iloc[p], PAGES)
        left, top, right, bottom = box
        for k in range(PER_PAGE):
            angle = random.uniform(-MAX_ANGLE, MAX_ANGLE)
            x = random.randint(left, right - CUT)
            y = random.randint(top, bottom - CUT)
            square = image[y:y + CUT, x:x + CUT]
            turned = turn(square, angle)
            edge = (len(turned) - CROP) // 2
            crops[i] = turned[edge:edge + CROP, edge:edge + CROP]
            angles[i] = angle
            i += 1
    return crops, angles

start = time.time()
train_crops, train_angles = make_examples(train)
validation_crops, validation_angles = make_examples(validation)
print(len(train_crops), "examples to learn from,", len(validation_crops), "to check on, made in", round(time.time() - start, 1), "s")
print("each example:", CROP, "×", CROP, "pixels and one angle; the first angles:", train_angles[:4].round(1))

In [ ]:
# Twelve of the examples with their labels.
pick = random.sample(range(len(train_crops)), 12)
show([train_crops[i] for i in pick], titles=[f"{train_angles[i]:+.1f}°" for i in pick], width=2.2)

## 5. The network

A convolutional network is made of filters like those in notebook 01, but nobody types their
numbers: they start random, and training changes them. This network has three layers of 3 × 3
filters; after each, only the strongest answer in every 2 × 2 block is kept, which halves the
picture. Two plain layers then make one number of what is left: the angle.

In [ ]:
# The network's starting numbers are random; the seed makes them the same on every run.
torch.manual_seed(SEED)

def new_network():
    """A network with random starting numbers: a square of CROP × CROP pixels goes in, one number comes out."""
    return torch.nn.Sequential(
        torch.nn.Conv2d(1, 8, kernel_size=3, padding=1),    # 8 filters of 3 × 3 on the square
        torch.nn.ReLU(),                                     # negative answers become 0
        torch.nn.MaxPool2d(2),                               # the strongest answer in each 2 × 2: 80 × 80 becomes 40 × 40
        torch.nn.Conv2d(8, 16, kernel_size=3, padding=1),   # 16 filters, each looking at all 8 answers
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(2),                               # 40 × 40 becomes 20 × 20
        torch.nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 32 filters on the 16 answers
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(2),                               # 20 × 20 becomes 10 × 10
        torch.nn.Flatten(),                                  # 32 maps of 10 × 10: 3200 numbers in a row
        torch.nn.Linear(3200, 32),                           # 32 weighted sums of the 3200
        torch.nn.ReLU(),
        torch.nn.Linear(32, 1),                              # one weighted sum of the 32: the angle
    )

network = new_network()
sizes = [part.numel() for part in network.parameters()]   # how many numbers each part holds
print(sum(sizes), "numbers to learn")

In [ ]:
# The network reads numbers around 0 and answers with one: ink becomes about +1, paper about -1,
# and an answer of 1 means MAX_ANGLE degrees.
def as_input(crops):
    """The squares as the network reads them: ink near +1, paper near -1, one grey layer each."""
    numbers = (128 - crops.astype(np.float32)) / 128
    return torch.from_numpy(numbers).reshape(len(crops), 1, CROP, CROP)

def predict(network, crops):
    """The network's angle for every square, in degrees."""
    with torch.no_grad():                  # only answer; nothing is learnt here
        answers = network(as_input(crops))
    return answers[:, 0].numpy() * MAX_ANGLE

# The twelve examples of section 4, before any training.
guesses = predict(network, train_crops[pick])
titles = [f"label {train_angles[i]:+.1f}°\nguess {guess:+.1f}°" for i, guess in zip(pick, guesses)]
show([train_crops[i] for i in pick], titles=titles, width=2.2)

Untrained, the network gives every square about the same answer: its numbers are still random,
and nothing in them answers to the slope of the lines yet.

## 6. Training: guess, compare, adjust

The loop below is the whole of supervised learning, the loop of notebook 02 with about a hundred
thousand numbers to adjust instead of two. It takes 64 examples, lets the network guess
their angles, compares the guesses with the labels, and moves every one of the network's numbers a
little in the direction that makes the difference smaller. One round through all the training
examples is called an epoch. After each epoch the cell measures how far off the network is, on the
training examples and on the validation examples, which it never learns from.

In [ ]:
EPOCHS = 10   # rounds through all the training examples
BATCH = 64    # examples per step

def train_network(network, crops, angles):
    """Train the network on the squares and their angles. Returns its mean error in degrees after
    each epoch, on these examples and on the validation examples."""
    optimiser = torch.optim.Adam(network.parameters(), lr=0.001)   # Adam: a common way of taking the small steps
    inputs = as_input(crops)
    targets = torch.from_numpy(angles / MAX_ANGLE)                # the labels as numbers around 0, like the answers
    train_error = np.zeros(EPOCHS)
    validation_error = np.zeros(EPOCHS)
    start = time.time()
    for epoch in range(EPOCHS):
        order = random.sample(range(len(crops)), len(crops))      # all the examples, in a new order
        for i in range(0, len(crops), BATCH):
            batch = order[i:i + BATCH]
            guesses = network(inputs[batch])[:, 0]                  # guess
            loss = ((guesses - targets[batch]) ** 2).mean()         # compare: the mean squared difference
            optimiser.zero_grad()                                   # forget the directions of the last step
            loss.backward()                                         # which way each number should move
            optimiser.step()                                        # adjust: a small step that way
        train_error[epoch] = abs(predict(network, crops) - angles).mean()
        validation_error[epoch] = abs(predict(network, validation_crops) - validation_angles).mean()
        print(f"epoch {epoch + 1}: off by {train_error[epoch]:.1f}° on the training examples, "
              f"{validation_error[epoch]:.1f}° on the validation examples ({time.time() - start:.0f} s)")
    return train_error, validation_error

train_error, validation_error = train_network(network, train_crops, train_angles)

In [ ]:
def plot_losses(train_error, validation_error, title):
    """The two errors after each epoch."""
    epochs = range(1, EPOCHS + 1)
    plt.figure(figsize=(6, 3.5))
    plt.plot(epochs, train_error, marker="o", label="training examples")
    plt.plot(epochs, validation_error, marker="o", label="validation examples")
    plt.ylim(bottom=0)
    plt.xlabel("epoch")
    plt.ylabel("off by, in degrees (mean)")
    plt.title(title)
    plt.legend()
    plt.show()

plot_losses(train_error, validation_error, f"trained on {len(train)} pages")

Training makes the squared difference small, because squaring weighs the large mistakes most; the
plot shows the plain difference in degrees, which is easier to read.

## 7. Straighten the six scans

Now the test: the six crooked scans, from letters the network has never seen. Several squares are
cut out of a scan, and the network predicts the angle of each. The median of these answers is
then the answer for the whole page. This makes sure that the page is considered as a whole, and
not only a small part of it. You can experiment with the number of squares below.

In [ ]:
def scan_crops(scan, n):
    """n squares of CROP pixels from random spots inside the text of a scan, shrunk by SCALE like the pages."""
    image, box = scaled_page(scan, CROOKED)
    left, top, right, bottom = box
    crops = np.zeros((n, CROP, CROP), dtype=np.uint8)
    for k in range(n):
        x = random.randint(left, right - CROP)
        y = random.randint(top, bottom - CROP)
        crops[k] = image[y:y + CROP, x:x + CROP]
    return crops

crops = scan_crops(scans.iloc[0], 12)
guesses = predict(network, crops)
show(crops, titles=[f"{guess:+.1f}°" for guess in guesses], width=2.2)
print(f"scan-01: the median of the twelve answers is {np.median(guesses):+.1f}°")

In [ ]:
# Your turn: with one square, the answer depends on where the square falls. The cell asks three
# times; raise n and watch how close the three medians come.
n = 1
for attempt in range(3):
    guesses = predict(network, scan_crops(scans.iloc[0], n))
    print(f"try {attempt + 1} with n = {n}: the median is {np.median(guesses):+.1f}°")

In [ ]:
PER_SCAN = 200   # squares from each scan: enough for the median to hold still
for i in range(len(scans)):
    scan = scans.iloc[i]
    guesses = predict(network, scan_crops(scan, PER_SCAN))
    angle = np.median(guesses)
    straight = turn(scan_images[i], -angle)   # turned back by the opposite angle
    show([scan_images[i], straight], titles=[f"{scan['file']} as scanned", f"turned by {-angle:+.1f}°"], width=4)

Look at each pair: are the lines of writing level now? Judge by the lines, not by the edges of
the sheet. The sheets lay straight in the scanner and the writing sloped on them, so once the
writing is level, the sheet is tilted.